# Extract data from CERCA raw files

In [1]:
from docx import Document
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [2]:
interest_centers = ['ISGlobal']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'

## Extract DOI list

**Each file has a different format so we go center by center**

### ISGlobal

In [3]:
center_name = 'ISGlobal/'
file_name = 'ISGlobal_DOIs Publications_2021-2024'

In [4]:
df_ISGlobal = []
for year in range(2021, 2025):
    df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', sheet_name = str(year), skiprows = 1)
    df_ISGlobal.append(df[['Doi']])
df_ISGlobal = pd.concat(df_ISGlobal, ignore_index=True).rename(columns = {'Doi' : 'DOI'} )
df_ISGlobal['Center'] = 'ISGlobal'
df_ISGlobal

,DOI,Center
0,10.1038/s41380-019-0558-2,ISGlobal
1,10.1016/j.edumed.2019.09.004,ISGlobal
2,10.1038/ng.2238,ISGlobal
3,10.1111/all.14422,ISGlobal
4,10.1016/j.nrl.2020.02.006,ISGlobal
...,...,...
2820,10.1016/j.aohep.2023.101133,ISGlobal
2821,10.1016/j.eimce.2022.12.016,ISGlobal
2822,10.1016/j.eimc.2022.12.009,ISGlobal
2823,10.1038/s41370-022-00453-6,ISGlobal


In [5]:
df_centers = df_ISGlobal
df_centers

,DOI,Center
0,10.1038/s41380-019-0558-2,ISGlobal
1,10.1016/j.edumed.2019.09.004,ISGlobal
2,10.1038/ng.2238,ISGlobal
3,10.1111/all.14422,ISGlobal
4,10.1016/j.nrl.2020.02.006,ISGlobal
...,...,...
2820,10.1016/j.aohep.2023.101133,ISGlobal
2821,10.1016/j.eimce.2022.12.016,ISGlobal
2822,10.1016/j.eimc.2022.12.009,ISGlobal
2823,10.1038/s41370-022-00453-6,ISGlobal


## Check which publications are not in OA using DOI

In [6]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [9]:
in_query = str(tuple(df_centers.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wa.author_position,
       wa.is_corresponding,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """
df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.1289/ehp8117,Yoav Levi,7,middle,False,4210100416,IL
1,10.1038/s41588-021-00786-2,Tomislav Kuliš,192,middle,False,138937732,HR
2,10.1038/s41588-021-00786-2,Tomislav Kuliš,192,middle,False,181343428,HR
3,10.1111/all.14422,Mario Sánchez‐Borges,249,middle,False,4210140747,VE
4,10.1186/s13054-024-05180-y,Cristian C. Serrano-Mayorga,2,middle,False,157650460,CO
...,...,...,...,...,...,...,...
85978,10.1038/s41467-021-24673-w,Linda‐Gail Bekker,7,middle,False,138025133,ZA
85979,10.1038/s41380-020-00976-0,Nastassja Koen,17,middle,False,157614274,ZA
85980,10.1371/journal.pone.0248538,Fredros O. Okumu,8,last,False,192619145,ZA
85981,10.1038/s41572-023-00452-3,Linda‐Gail Bekker,1,first,True,157614274,ZA


### Identify CERCA authors using OA (93% of the dataset)

- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation [2k affiliations]
  - String search with keywords for = 1 doi per raw affilation [4k affilations]

**We identify 70% of the provided DOI's**

For the ResearchMar the retrieval is 61%; being the center with most publications we will force the manual identification to increase it by selecting in a second phase using a manual validation [word mar and Barcelona]. We reach 67%

The problem is the hypothesis of using the parent affiliations (problem for the hospital) that is not good enough but I can't manually review all the affiliations of spain. So I can't improve it further

So to solve it we select the not found dois in Researchmar using the previous parent affiliations hypothesis and chec the raw affiliations in Spain looking for strings (mar, imim, parc combined with barcelona without checking).We reach 84%

In [10]:
cerca_centers = {'BETA' : [''], # NOT IN OA
                    'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'BETA' : ['115304662'],
                    'CREAF' : ['123044942', '71999127'],
                    'ICN2' : ['123044942'],
                    'ISGlobal' : ['123044942', '170486558'],
                    'ResearchMar' : ['170486558']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(df_OA.institution_id.isin(parents_cerca)) & (~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
24431,10.1111/jcmm.16736,Raquel Rabionet,44,middle,False,71999127,ES
24441,10.3389/fpubh.2021.592500,Ada Melo-Vallès,1,first,False,170486558,ES
24446,10.1128/spectrum.02142-23,José Antonio Martínez,18,middle,False,71999127,ES
24452,10.1007/s40121-021-00445-3,José Antonio Martínez,12,middle,False,71999127,ES
24459,10.1080/22221751.2024.2392659,Gemina Santana,14,middle,False,71999127,ES
...,...,...,...,...,...,...,...
57411,10.1016/j.eimc.2023.02.004,Elena Sulleiro,17,middle,False,123044942,ES
57427,10.1097/hep.0000000000000545,Ramón Bataller,12,middle,False,71999127,ES
57494,10.3390/nu16070974,Ariadna Pinar-Martí,1,first,False,170486558,ES
57529,10.3390/nu14030518,Ariadna Pinar-Martí,3,middle,False,170486558,ES


In [13]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1158/1055-9965.epi-23-0453,William R. Bamlet,"30Department of Quantitative Health Sciences, ...",1330342723,US
1,10.1038/s41598-021-98296-y,None,"ISGlobal, Hospital Clínic, Universitat de Barc...",71999127,ES
2,10.1038/s41586-023-06355-3,Elizabeth Woodward,"Poole Hospital, Poole, UK",2800892894,GB
3,10.1016/j.jhep.2023.10.007,Núria Cañete,"Liver Section, Gastroenterology Department and...",4210130874,ES
4,10.1038/s41586-023-06355-3,Catherine H. Weldon,"23andMe, Sunnyvale, CA, USA",4210097491,US
...,...,...,...,...,...
14087,10.1016/j.envpol.2024.123612,Élise Bannier,Centre Hospitalier Universitaire de Rennes [CH...,4210155724,FR
14088,10.1038/s41391-021-00446-w,Lluís Cecchini,"Urology Department, Hospital del Mar-IMIM, Aut...",123044942,ES
14089,10.1016/j.tmaid.2021.101985,María‐Jesús Pinazo,Consorcio de Investigación Biomédica en Red de...,71999127,ES
14090,10.1016/j.jaip.2024.06.040,Jose E. Gereda,"Clínica Ricardo Palma, Allergy & Immunology De...",4210120426,PE


In [16]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
to_check.to_csv('to_check_ISGlobal.csv')
to_check

660


,DOI
raw_affiliation,
"Universitat Pompeu Fabra (UPF), Barcelona, Spain",105
"ISGlobal, Barcelona, Spain",96
"Barcelona Institute for Global Health (ISGlobal), Barcelona, Spain",45
"CIBER Epidemiología y Salud Pública (CIBERESP), Madrid, Spain",34
"IMIM (Hospital del Mar Medical Research Institute), Barcelona, Spain",29
...,...
"Department of Clinical Sciences, Faculty of Medicine, University of Barcelona, Barcelona, Spain.",2
"Universitat de Barcelona (UB), Barcelona, Spain",2
"Unidad de Infección Viral e Inmunidad, Centro Nacional de Microbiología (CNM), Instituto de Salud Carlos III (ISCIII), Madrid, Spain",2


In [14]:
df_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '>1ISGlobal', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


0.8123177842565598

In [17]:
cerca_string = {'ISGlobal' : ['global']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_check_2 = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check.raw_affiliation))].reset_index()
df_check_2['raw_affiliation'] = df_check_2['raw_affiliation'].str.lower()

df_check_2['CERCA'] = df_check_2['raw_affiliation'].str.contains(('|'.join(cerca_string)), case=False, na=False).map({True: 'TRUE', False: 'FALSE'})
df_check_2

,raw_affiliation,DOI,CERCA
0,"institute of neurosciences, university of barc...",1,FALSE
1,"institute of neurosciences, university of barc...",1,FALSE
2,"institute of neurosciences, universitat de bar...",1,FALSE
3,"liver unit, hospital universitario puerta de h...",1,FALSE
4,"liver transplant unit, hospital clinic – idiba...",1,TRUE
...,...,...,...
4244,"department of microbiology, hospital clinic of...",1,FALSE
4245,"department of microbiology, hospital clinic of...",1,FALSE
4246,"department of microbiology, hospital clinic of...",1,FALSE
4247,"department of microbiology, hospital clinic of...",1,TRUE


In [18]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.8163265306122449

In [19]:
df_check = pd.concat((cerca_authors, check_1, check_2))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
24515,10.1080/19490976.2024.2415488,Andrea Aira,2,middle,False,4210148332,ES,NaN
24533,10.1016/j.isci.2022.105765,Yunuen Ávalos-Padilla,1,first,False,4210148332,ES,NaN
24541,10.1016/j.jconrel.2021.01.028,Yunuen Ávalos-Padilla,8,middle,False,4210148332,ES,NaN
24542,10.1128/mbio.01636-21,Anastasia K. Pickford,1,first,False,4210148332,ES,NaN
24552,10.1186/s12915-022-01374-4,Yunuen Ávalos-Padilla,2,middle,False,4210148332,ES,NaN
...,...,...,...,...,...,...,...,...
14043,10.1016/j.recesp.2021.06.030,Jeroen de Bont,<NA>,NaN,<NA>,4210148332,ES,"ISGlobal, Barcelona, España"
14057,10.1002/alz.13862,Eleni Palpatzis,<NA>,NaN,<NA>,4210128274,ES,"ISGlobal, Barcelona Institute of Global Health..."
14064,10.1128/spectrum.02628-22,Montserrat Gállego,<NA>,NaN,<NA>,4210148332,ES,"ISGlobal, Barcelona, Hospital Clínic, Universi..."
14074,10.1016/j.ajog.2022.07.027,Jordi Bosch,<NA>,NaN,<NA>,4210148332,ES,"Department of Microbiology, Biomedical Diagnos..."


In [20]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_final = df_OA.merge(df_centers, on = 'DOI').drop_duplicates().reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_ISGlobal.csv', index = False)
df_final

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1289/ehp8117,Yoav Levi,7,middle,False,4210100416,IL,False,ISGlobal
1,10.1038/s41588-021-00786-2,Tomislav Kuliš,192,middle,False,138937732,HR,False,ISGlobal
2,10.1038/s41588-021-00786-2,Tomislav Kuliš,192,middle,False,181343428,HR,False,ISGlobal
3,10.1111/all.14422,Mario Sánchez‐Borges,249,middle,False,4210140747,VE,False,ISGlobal
4,10.1186/s13054-024-05180-y,Cristian C. Serrano-Mayorga,2,middle,False,157650460,CO,False,ISGlobal
...,...,...,...,...,...,...,...,...,...
85978,10.1038/s41467-021-24673-w,Linda‐Gail Bekker,7,middle,False,138025133,ZA,False,ISGlobal
85979,10.1038/s41380-020-00976-0,Nastassja Koen,17,middle,False,157614274,ZA,False,ISGlobal
85980,10.1371/journal.pone.0248538,Fredros O. Okumu,8,last,False,192619145,ZA,False,ISGlobal
85981,10.1038/s41572-023-00452-3,Linda‐Gail Bekker,1,first,True,157614274,ZA,False,ISGlobal
